In [ ]:
!pip install Biopython #Si no se tiene la librería Biopython

from Bio import Entrez
Entrez.email = "user@gmail.com" # Siempre hay que identificarse 

#Primero es importante saber: qué podemos buscar con Entrez en la base de datos nucleotide? (Se puede aplicar a todas las db)

stream = Entrez.einfo (db="assembly")
record = Entrez.read(stream)

for field in record["DbInfo"]["FieldList"]:
    print("%(Name)s, %(FullName)s, %(Description)s" % field)

In [ ]:
# Ahora que ya sabemos los campos buscaremos secuencias de blaZ en S. aureus de manera específica

query = "(blaZ[Gene Name] OR beta-lactamase[Title]) AND Staphylococcus aureus[Organism] AND complete cds[Title]"

stream = Entrez.esearch(db="nucleotide", 
                        term=query,
                        RetMax=50) #Si no se pone parece que por defecto pone 20
record = Entrez.read(stream) 
# Es mejor Entrez.read que no handle.read, porque esta es la funcion de parsing 
# de Entrez, y lo que hace es convertirlo en un diccionario que pueda interpretar Python

stream.close()

print(f"Secuencias encontradas: {record['Count']}")

Secuencias encontradas: 40


In [ ]:
#Ahora que sabemos lo que hay, lo descargaremos con el módulo EFetch de Entrez
#Todos los parámetros están  en https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.EFetch

stream = Entrez.efetch(db="nucleotide",
                       id=record['IdList'],
                       rettype="fasta",
                       retmode="text")

# The open() function opens a file, and returns it as a file object. Si este archivo no existe lo crea!
# La estructura es open("file", mode). Como modo hemos puesto w = write, para que escriba lo que sea que le hemos dicho debajo
# https://www.w3schools.com/python/ref_func_open.asp
 
with open("blaZ_sequences.fasta", "w") as output_file:
    output_file.write(stream.read())
stream.close()

#Esto nos ha creado un fichero llamado "blaZ_sequences" donde estan todas las secuencias que se ajustan a la búsqueda anterior que hemos hecho

In [ ]:
# Si vemos el fichero, nos damos cuenta que no todas las secuencias se ajustan solo a blaZ. Tenemos que limpiarlas un poco
# Para definir el rango del filtro primero tendremos que ver una pequeña descripcion de nuestras sec

from Bio import SeqIO # Libreria para leer, escribir y manipular archivos de secuencias biológicas (FASTA, GenBank)

for secuencia in SeqIO.parse("blaZ_sequences.fasta", "fasta"):
    print(f"{secuencia.id}, {len(secuencia)} pb, {secuencia.description}")

# Al correr esto vemos que aprox. las que estan entre 900 - 1350 bp (U58139.2) son b-lactam solas, pero las que superan este numero
# contienen otros elementos. Estas secuencias no las queremos.

In [ ]:
#Sabiendo esto, vamos a crear otro documento con las secuencias filtradas:

# Aqui NO usamos el comando open() porque este es propio de python y no entiende el formato fasta. Tendriamos que escribir mucho código adicional
# Para eso nos sirve SeqIO, y antes no lo hemos usado pq habia que descargar las secuencias para empezar. Las principales diferencias entre los 2:

# open() + efetch: para descargar datos de internet y guardarlos en bruto como texto
# SeqIO: para leer y manipular archivos de secuencias que ya tienes guardados

secuencias_filtradas = [] # Creamos una lista para nuestras secuencias definitivas
secuencias_largas =  [] # Y otra para las que tengamos que filtrar manualmente

for secuencia in SeqIO.parse("blaZ_sequences.fasta", "fasta"): # El segundo "fasta" es para decirle que formato es, porque solo no lo puede leer
    if len(secuencia) <= 1400:
        secuencias_filtradas.append(secuencia)
        SeqIO.write(secuencias_filtradas, "blaZ_filtradas.fasta", "fasta") 
    else:
        secuencias_largas.append(secuencia)
        SeqIO.write(secuencias_largas, "blaZ_filtradas_largas.fasta", "fasta") # Las guardamos en otro archivo fasta

# Después de hacer un filtrado manual de las secuencias largas miramos cuántas han quedado al final:

print(f"Total de secuencias: {len(list(SeqIO.parse('blaZ_filtradas.fasta', 'fasta')))}")

22